# Prompt Engineering en Turismo

En este notebook se analiza el impacto de diferentes técnicas de Prompt Engineering en las respuestas generadas por un modelo de lenguaje. Para ello, se utiliza Gemini mediante su API y un conjunto de atractivos turísticos. Se comparará un prompt básico con diferentes estrategias de diseño de prompts, evaluando la calidad, estructura y consistencia de las respuestas obtenidas.

# Promp engineering con Gemini


## 1. Importación de librerías

In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [ ]:
import os
import json
import time
import re
import pandas as pd

from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import JSON

## 2. Conexión al modelo

Cargar las api keys para acceder a los modelos

In [ ]:
GEMINI_API_KEY = userdata.get("GEMINI_API")
client = genai.Client(
    api_key=GEMINI_API_KEY
)

In [ ]:
GEMINI_API_KEY_2 = userdata.get("GEMINI_API_2")
client_2 = genai.Client(
    api_key=GEMINI_API_KEY_2
)

In [ ]:
GEMINI_API_KEY_ACCOUNT_2 = userdata.get("GEMINI_API_A2")
client_3 = genai.Client(
    api_key=GEMINI_API_KEY_ACCOUNT_2
)

### 2.1 Probar que el modelo esta funcionando


In [ ]:
response = client_2.models.generate_content(
    model="gemini-3.6-flash",
    contents="Responde únicamente con la palabra: Funcionando")

print(response.text)

Funcionando


In [ ]:
response = client_3.models.generate_content(
    model="gemini-3.6-flash",
    contents="Responde únicamente con la palabra: Funcionando")

print(response.text)

Funcionando


## 3. Funciones para consultar

### 3.1 Guardar json

In [ ]:
def guardar_json_gemini(datos, nombre_atraccion):
    """
    Guarda los datos de Gemini en un archivo JSON
    dentro del directorio actual del notebook.
    """

    # Crear un nombre de archivo seguro
    nombre_archivo = nombre_atraccion.lower()
    nombre_archivo = re.sub(r"[^a-z0-9áéíóúüñ]+", "_", nombre_archivo)
    nombre_archivo = nombre_archivo.strip("_")

    ruta_json = f"{nombre_archivo}.json"

    with open(ruta_json, "w", encoding="utf-8") as archivo:
        json.dump(
            datos,
            archivo,
            ensure_ascii=False,
            indent=4
        )

    return ruta_json

### 3.2 Limpia el json

In [ ]:
def limpiar_json_respuesta(texto):
    """
    Limpia la respuesta de Gemini cuando viene envuelta en bloques ```json.
    """
    if not texto:
        return ""

    texto = texto.strip()
    texto = texto.replace("```json", "")
    texto = texto.replace("```", "")
    texto = texto.strip()

    return texto

### 3.3 Consuta en el modelo

In [ ]:
def consultar_gemini(
    prompt,
    guardar_json=False,
    nombre_atraccion=None,
    max_intentos=4):
    """
    Consulta Gemini.

    guardar_json=False:
        devuelve la respuesta como texto.

    guardar_json=True:
        intenta convertir la respuesta a JSON y la guarda.
    """

    for intento in range(max_intentos):
        try:
            response = client_3.models.generate_content(
                #model="gemini-2.5-flash", #Para el modelo 1
                model="gemini-3.6-flash", #Para el modelo 2
                contents=prompt,
                config=types.GenerateContentConfig(
                    tools=[
                        types.Tool(
                            google_search=types.GoogleSearch()
                        )
                    ]
                )
            )

            #RESPUESTA NORMAL
            if not guardar_json:
                return response.text

            #RESPUESTA JSON
            texto = limpiar_json_respuesta(response.text)

            datos = json.loads(texto)

            ruta_json = guardar_json_gemini(
                datos,
                nombre_atraccion
            )

            datos["_archivo_json"] = ruta_json

            return datos

        except json.JSONDecodeError:
            print("La respuesta no pudo convertirse a JSON.")
            return None

        except Exception as e:

            if "503" in str(e):

                if intento < max_intentos - 1:
                    espera = 2 ** intento

                    print(
                        f"Gemini está saturado. "
                        f"Reintentando en {espera} segundos..."
                    )

                    time.sleep(espera)

                else:
                    print("Gemini continúa sin estar disponible.")
                    return None

            else:
                print(e)
                return None

In [ ]:
respuesta = consultar_gemini("¿Cuál es la capital del estado de Tamaulipas?")

print(respuesta)

429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}
None


## 4. Variables

In [ ]:
nombre_atraccion = "Playa Miramar"
municipio = "Ciudad Madero"
estado = "Tamaulipas"

## 5. Pruebas

### Prueba: Sin Prompt Engineering estructurado

In [ ]:
prompt_basico = f"""
Busca información en internet sobre {nombre_atraccion}, ubicada en {municipio}, {estado}, México. Dame información turística sobre este lugar.
"""

In [ ]:
respuesta_basica = consultar_gemini(
    prompt_basico,
    guardar_json=False
)

print(respuesta_basica)

¡Excelente elección! Playa Miramar es uno de los destinos de playa más populares y queridos del noreste de México, conocida por su belleza, tranquilidad y un toque de misterio. Aquí tienes una guía turística completa:

---

## Playa Miramar: El Encanto del Golfo en Ciudad Madero, Tamaulipas

**Playa Miramar** es la joya turística de Ciudad Madero y el área metropolitana de Tampico. Ubicada en la costa del Golfo de México, es famosa por sus amplias extensiones de arena dorada, aguas tranquilas y una infraestructura turística bien desarrollada, ideal para familias, parejas y aventureros.

### Ubicación y Acceso

*   **Ciudad:** Ciudad Madero, Tamaulipas, México.
*   **Proximidad:** Se encuentra a pocos minutos de Tampico, lo que la hace muy accesible desde el Aeropuerto Internacional de Tampico (General Francisco Javier Mina).
*   **Cómo llegar:** Puedes llegar en coche particular, taxi o transporte público desde Tampico y Ciudad Madero. La mayoría de los hoteles en la zona ofrecen fácil

El primer prompt proporciona únicamente la tarea general y el contexto de la atracción turística, dejando al modelo libertad para decidir qué información
incluir y cómo presentarla. Se obtiene una respuesta amplia en lenguaje natural que incluye información turística relevante, pero también datos adicionales no solicitados. Además, la información no presenta una estructura estandarizada que permita su procesamiento automático.

### Prompt Engineering sin estructura

In [ ]:
prompt_intermedio = f"""
Busca información  en internet sobre la siguiente atracción turística:

Nombre: {nombre_atraccion}
Municipio: {municipio}
Estado: {estado}
País: México

Necesito obtener únicamente la siguiente información:

- Nombre
- Descripción
- Municipio
- Dirección
- Horario
- Servicios
- Si es gratis
- Si acepta mascotas
- Si es accesible para personas con discapacidad
- Calificación promedio
- Fuentes

Organiza la respuesta utilizando estos campos y evita incluir
información que no haya sido solicitada.
"""

In [ ]:
respuesta_intermedia = consultar_gemini(prompt_intermedio)

print(respuesta_intermedia)

Aquí tienes la información solicitada sobre Playa Miramar:

*   **Nombre:** Playa Miramar
*   **Descripción:** Playa Miramar es una de las atracciones turísticas más importantes y visitadas del noreste de México, ubicada en la costa del Golfo de México. Es conocida por su fina arena dorada, sus aguas tranquilas y su extensa oferta de servicios. Es un destino ideal para el esparcimiento familiar, la natación y la práctica de deportes acuáticos. En su extremo sur se encuentran las famosas escolleras de Playa Miramar, un punto donde las aguas del mar se encuentran con la desembocadura del río Pánuco, ofreciendo un lugar popular para la observación de delfines. También cuenta con el "Corredor Turístico de las Dunas Doradas".
*   **Municipio:** Ciudad Madero
*   **Dirección:** Avenida Tamaulipas, Ciudad Madero, Tamaulipas, México. (La playa se extiende a lo largo de la costa de Ciudad Madero, siendo la Avenida Tamaulipas la principal vía de acceso y referencia).
*   **Horario:** La playa es

### Prompt Engineering bajo una estructura

A partir de las limitaciones observadas en el primer experimento, se diseñó un
nuevo prompt incorporando instrucciones específicas, restricciones, definición
del formato de salida y reglas para la normalización de los datos. El objetivo es obtener una respuesta estructurada que pueda ser procesada
automáticamente mediante Python.

In [ ]:
prompt_estructurado = f"""
    Busca información en internet sobre la siguiente atracción turística:

    Nombre de la atracción: {nombre_atraccion}
    Municipio: {municipio}
    Estado: {estado}
    País: México

    Necesito que devuelvas únicamente un JSON válido, sin explicación adicional.

    Estructura obligatoria:

    {{
        "nombre": "",
        "descripcion": "",
        "municipio": "",
        "direccion": {{
            "calle": "",
            "numero": "",
            "colonia": "",
            "codigo_postal": ""
        }},
        "ubicacion": {{
            "latitud": null,
            "longitud": null
        }},
        "horario": {{
            "dias": [],
            "hora_apertura": "",
            "hora_cierre": ""
        }},
        "servicios": [],
        "es_gratis": null,
        "es_petfriendly": null,
        "es_accesible": null,
        "calificacion_promedio": null,
        "fuentes": []
    }}

    Reglas:
    -No inventes datos.
    -Si no encuentras un dato, usa null, cadena vacía o lista vacía.
    -En "servicios" incluye servicios turísticos concretos, por ejemplo:
        estacionamiento, sanitarios, áreas verdes, visitas guiadas, restaurante,
        venta de alimentos, accesibilidad, juegos infantiles, mirador, senderos,
        renta de equipo, zona de descanso, información turística, etc.
    -En "dias" usa nombres sin acentos y en formato compatible con individuos OWL:
        Lunes, Martes, Miercoles, Jueves, Viernes, Sabado, Domingo.
    -En horarios usa formato HH:MM:SS, en formato de 24 horas.
    -En valores booleanos usa true, false o null.
    -En "fuentes" agrega URLs o nombres de sitios consultados cuando estén disponibles.
    -La descripción debe ser breve y útil para una ficha turística, entre 70 y 100 palabras (unos 500 caracteres).
    """

In [ ]:
respuesta_estructurada = consultar_gemini(
    prompt_estructurado,
    guardar_json=True,
    nombre_atraccion=nombre_atraccion
)

respuesta_estructurada

429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}


In [ ]:
json_file_path = "/content/sample_data/Playa_Miramar_20260622_133050.json"
try:
    with open(json_file_path, "r", encoding="utf-8") as f:
        playa_data = json.load(f)
    print("Json Cargado")
except FileNotFoundError:
    print(f" No se encontró la dirección '{json_file_path}'.")

Json Cargado


In [ ]:
JSON(playa_data)

<IPython.core.display.JSON object>